# 07｜健身房會員：三階提問實作

[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/ai-excel-analytics/blob/main/03_AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90/07_gym_groupby/lesson.ipynb)

> 📌 **資料情境**：checkins.csv（進場刷卡紀錄）、members.csv（會員基本資料）；一列代表 **一次進場紀錄（注意：不是一位會員）**。

| ① 看懂資料 | ② 一題一圖（人機選圖） | ③ 解讀數據與圖表 | ④ 交付報告 |
| :--- | :--- | :--- | :--- |
| 欄位角色・邊界檢核 | 建議優缺點 ➔ 決定圖表 ➔ 繪圖 | 核心結論・數據洞察・說出限制 | RTCCF 框架產出 Word / Excel |

**你負責提問與驗收，AI 負責寫程式。** 執行第一格載入資料與繁中字型後，依序將三階 prompt 貼給 Colab Gemini 或支援 Notebook 的 AI 對話框。所有程式碼、數字表、圖表與解讀完全由 AI 產出，你不需要手寫任何程式碼。

**官方範例核心問題**：哪家分店的入場人次最多？與不重複會員人數有何差異？（你也可以在第一階由 AI 提出的 5 個問題中自由挑選，重複練習第 2 階與第 3 階！）

In [ ]:
# 第 0 步：下載範例資料並安裝繁體中文字型（請先點擊左側 ▶ 執行此儲存格）
import os
from pathlib import Path
from urllib.request import urlretrieve

# 1. 下載資料（已存在就不重複下載；若下載失敗，請手動上傳到 data/）
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
files = {
    'gym_visits.csv': 'https://raw.githubusercontent.com/roberthsu2003/ai-excel-analytics/main/03_AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90/07_gym_groupby/data/gym_visits.csv',
}
for name, url in files.items():
    if not (data_dir / name).exists():
        urlretrieve(url, data_dir / name)
print('📂 data/ 內的檔案：', sorted(p.name for p in data_dir.iterdir()))

# 2. 安裝與設定 Colab 繁體中文字型 (思源黑體)
%pip install -q mplfonts
import glob
import mplfonts
from matplotlib import font_manager
from mplfonts import use_font

for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
    font_manager.fontManager.addfont(f)
use_font("Noto Sans CJK SC")

print("✅ 環境初始化完成！資料已就緒，繁體中文字型已設定成功。請開始進行下方單元。")

---

## ① 第一階｜看懂資料

**思考重點與邊界提示：** 同一人會多次進場：「人次 (count)」和「不重複人數 (nunique)」是兩回事；人次高可能是少數會員頻繁來，不代表會員總數最多。

📋 **複製以下 Prompt 貼給 AI：**

```text
請讀取已載入/下載的資料（data/checkins.csv）：
1. 顯示前 5 列、總列數、所有欄位名稱與各欄缺值數
2. 說明「一列代表什麼」，並將所有欄位分成「指標」、「條件」、「標識」三種角色
3. 檢查資料的時間範圍、重複值或異常/缺值狀況
4. 根據資料欄位，提出 5 個可以回答的商業/分析問題（涵蓋不同面向），以及 1 個不能回答的問題並說明理由
⚠️ 先不要寫程式畫圖，也不要提前下推論或結論。
```

**你要檢查（驗收標準）：**
- [ ] 筆數與缺值數量符合原始資料
- [ ] 欄位角色分類正確（標識不可計算、指標才可運算）
- [ ] AI 提出的 5 個問題是否確實能由現有欄位回答

---

## ② 第二階｜一個問題、一張圖（人機協作選圖）

> 🎯 **人機協作流程：**
> 1. 第一階 AI 會提出 5 個可回答的問題，你可以**挑選其中 1 個**想探索的問題（或使用下方的官方範例問題）。
> 2. **先請 AI 建議適合的圖表並分析優缺點**，先不要急著畫圖。
> 3. 由你運用選圖規則做決策並**指定圖表類型**，再請 AI 產出排序數字表與繪圖。

### 步驟 2-1｜填入問題，請 AI 建議適合圖表與優缺點

📋 **複製給同一段 AI 對話（將問題替換為你想分析的問題）：**

```text
延續第一階，我想針對這個問題進行深入分析：
『＿＿＿＿＿＿＿＿＿＿＿＿』
（範例參考問題：哪家分店的入場人次最多？與不重複會員人數有何差異？）

請先不要寫程式畫圖，請先：
1. 說明這個問題屬於哪種問題類型（比較、趨勢、占比、分布、關聯）。
2. 建議 2～3 種適合呈現此問題的圖表類型，並條列分析每種圖表的「優點」與「缺點」（請考慮類別數量、名稱長短、數值特性等）。
3. 說明你最推薦哪一種圖表，以及推薦的理由是什麼。
等我確認要採用哪種圖表後，我們再進行下一步。
```

---

### 步驟 2-2｜決定圖表，請 AI 產出數字表與繪圖

📋 **看過 AI 建議後，由你決定圖表類型，再貼出以下 Prompt：**

```text
我決定採用【＿＿圖】（例如：垂直長條圖）。

請先產出排序好的統計數字表，再只畫一張【＿＿圖】。
繪圖注意事項：
1. 自動配置合適的坐標軸（若有數值軸請從 0 開始，避免截斷視覺比例）
2. 圖上清楚標註數值與單位，文字全部使用繁體中文（沿用已設定的字型）
3. 排除無效或缺值的資料，並符合邊界檢核防呆
⚠️ 一次只回答這個問題，不要畫其他無關圖表。
```

> 💡 **官方範例情境參考：** 若詢問「哪家分店的入場人次最多？與不重複會員人數有何差異？」，官方推薦使用【垂直長條圖】，理由：長條圖能直觀比較各分店總量；若同時呈現「入場人次」與「會員人數」，使用分組長條圖可清楚看出重度使用程度。

**你要檢查（驗收標準）：**
- [ ] 先核對數字表格的計算邏輯是否正確
- [ ] 圖上的數字與表格是否完全一致
- [ ] 數值軸是否從 0 開始、單位與中文字型是否正確顯示

---

## ③ 第三階｜解讀數據與圖表（說出洞察與限制）

沿用第二階產出的數字表與圖表，**留在同一段 AI 對話中**，請 AI 解釋分析結果。

📋 **複製給同一段 AI 對話：**

```text
請根據剛才產出的數字表與圖表，說明如何正確解釋這些分析結果。

解讀注意事項：
1. 核心結論：回答剛才設定的問題，並引用圖表中的具體數據支持
2. 數據洞察：說明數據與圖表呈現的重要現象、極端值或趨勢特徵
3. 資料限制：寫明統計範圍、分母，以及是否有被排除的資料（如缺考、異常值、未完成訂單）
4. 不能推論的事：說明單憑目前數據無法證明什麼（嚴守邊界：不把相關當因果、不妄下過度結論）
```

**你要檢查（驗收標準）：**
- [ ] 核心結論有具體數據支持，與圖表完全一致
- [ ] 若涉及比率或百分比，是否有清楚標明分母
- [ ] 沒有把相關說成因果，明確指出資料限制與邊界

> ✍️ **我的解讀筆記：**（雙擊此處，記錄下 AI 的哪點洞察最關鍵？指出了哪些限制？）

---

## ④ 第四階｜交付報告（選做：產出 RTCCF 提示詞）

> 🎯 **一鍵交付：** 完成前三階後，直接複製以下 Prompt 貼回 Colab，Colab AI 就會自動將所有成果打包成 RTCCF 提示詞。
> 複製該提示詞給具備檔案生成能力的外部 AI，即可直接生成 **Word 報告（.docx）** 或 **Excel 活頁簿（.xlsx）**！

### 選項 A｜產出「Word 報告專用」RTCCF 提示詞
```text
目前我們已經在對話中完成了資料分析與第三階解讀。
現在我想要請其他具備檔案生成能力的 AI 幫我生成正式的 Word 報告檔案（.docx），
請幫我撰寫一段符合 RTCCF 框架（Role, Task, Context, Constraints, Format）的 Markdown 提示詞，方便我直接複製使用。

請在該 RTCCF 提示詞中明確規範：
1. Role：資深商業數據分析顧問與專業排版專家。
2. Task：調用 Python 工具（如 python-docx 與 matplotlib），將分析成果產出為可下載的 Word 報告檔案（.docx）。
3. Context：自動填入我們剛才分析的資料背景、探討的問題，以及「第三階核實後的完整結論、數據洞察與限制」。
4. Constraints：嚴格依據報告（嚴禁腦補或把相關當因果）；內文必須插入剛才繪製的高畫質圖表與排版數據表；具備封面、摘要、各題分析與限制說明。
5. Format：直接提供產生該 .docx 檔案的完整 Python 程式碼或下載連結。
```

### 選項 B｜產出「Excel 活頁簿專用」RTCCF 提示詞
```text
目前我們已經在對話中完成了資料分析與第三階解讀。
現在我想要請其他具備程式執行能力的 AI 幫我建立專業的 Excel 活頁簿（.xlsx），
請幫我撰寫一段符合 RTCCF 框架（Role, Task, Context, Constraints, Format）的 Markdown 提示詞，方便我直接複製使用。

請在該 RTCCF 提示詞中明確規範：
1. Role：資深資料工程與 Excel 報表自動化專家。
2. Task：調用 Python 工具（如 openpyxl 或 xlsxwriter），建立結構化且含原生圖表的 Excel 檔案（.xlsx）。
3. Context：自動填入資料背景與剛才分析的核心問題。
4. Constraints：第 1 個工作表必須是「原始資料」；各問題建立獨立工作表（附統計表、Excel 原生圖表與第三階結論備註）。
5. Format：直接提供產生該 .xlsx 檔案的完整 Python 程式碼或下載連結。
```

---

[⬅️ 回範例目錄](../README.md)　｜　[🏠 回學習地圖](../../README.md)